<a href="https://colab.research.google.com/github/chaseshallow/Ansci4040FinalClassProject/blob/main/Step03aTrackTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 03a · Track the cow with SAM 2.1: test (4 minutes)

**SAM 2.1** follows the cow's **outline (mask)** through 1080–1320 s (the span step 02a covered), starting from the YOLO boxes of **Step 02a** (`Step02aDetectTest/Detections.csv`). It keeps following her where YOLO loses her (lying, straining), so she stays **one track** instead of dozens (the old ByteTrack tracker gave 47). Takes ~5–6 min on a T4.

**How:** the clip is cut into 2-minute chunks. Each chunk starts from where SAM left the cow at the end of the previous one (YOLO's box replaces it when YOLO is sure, to correct drift); the first chunk starts from the first confident YOLO box. SAM then follows her mask frame by frame, keeping the same track number. Every SAM box is checked against YOLO's:

- **Agrees** (`ok` in the preview): YOLO sees the cow in the same place
- **NoYoloBox** (`sam`): YOLO saw nothing, SAM kept following her
- **Disagrees** (`??`): YOLO sees a cow elsewhere but not where SAM is; SAM may have **drifted** (calf, floor): check the preview

**If it stops** (disconnect, GPU time used up): open the notebook again, even on another account, and run cells 1–3 with the **same settings**: finished chunks are kept in `Chunks/` on the Shared Drive and it continues from the next one. Otherwise it **overwrites**: a finished run, changed settings or new step 02 boxes start over.

Writes into the Shared Drive, `CalvingPenPipelineOutputs/<video>/Step03aTrackTest/`:

- `Tracks.csv`: one row per animal per frame: **input for step 04a**; `TrackSummary.csv`: one row per track
- `Comparison.csv`: ByteTrack baseline vs SAM on the same time span
- `PreviewTracks.mp4`: mask (coloured and outlined, the rest darkened), box and track ID, 4x speed
- `Chunks/`: per-chunk pieces, needed to resume

Needs a **T4 GPU**. The folder is shared: one person runs a step at a time. Edit cell 2 and rerun; don't add cells.

In [ ]:
# 1. Setup: mount Drive, get the pipeline code from GitHub, install SAM.
from google.colab import drive
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/chaseshallow/Ansci4040FinalClassProject.git"
RepoBranch = "main"
RepoDir = "/content/Ansci4040FinalClassProject"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
%pip install -q -r "{RepoDir}/CalvingPen/requirements.txt"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline.Track import TrackCows
from CalvingPipeline.Paths import OutputRoot
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

In [ ]:
# 2. Settings: the only cell to edit.
VideoStem = "2026-09-26 00-00-00 Calving Pen 2"
VideoOutputDir = OutputRoot / VideoStem   # Shared Drive: CalvingPenPipelineOutputs/<video>/
ClipPath = VideoOutputDir / "ReferenceClip.mp4"   # step 01
DetectionsDir = VideoOutputDir / "Step02aDetectTest"                        # step 02a
CompareTracksPath = VideoOutputDir / "BaselineByteTrack" / "Tracks.csv"   # optional: old tracker, to compare
OutputDir = VideoOutputDir / "Step03aTrackTest"

ChunkSeconds = 120     # SAM 2 memory grows per frame; restart it from a YOLO box every 2 min
SeedConfidence = 0.4   # YOLO box needed to start following a new animal (the cow itself is carried over between chunks)
Half = False           # True = half precision on the GPU: faster; check the result on the test first
Preview = True         # False = no preview video (a little faster)
PreviewSpeed = 4
Resume = True          # True = continue a stopped run (same settings); False = start over

assert (DetectionsDir / "Detections.csv").exists(), f"Run step 02a first: {DetectionsDir}"

In [ ]:
# 3. Run: track with SAM 2.1 (continues where a stopped run ended), then compare and show the preview.
import pandas as pd
from IPython.display import Video, display

Stats = TrackCows(ClipPath, DetectionsDir, OutputDir, ChunkSeconds=ChunkSeconds, SeedConfidence=SeedConfidence,
                  CompareTracksPath=CompareTracksPath, Half=Half, Preview=Preview, PreviewSpeed=PreviewSpeed,
                  Resume=Resume)

display(pd.read_csv(OutputDir / "Comparison.csv"))
display(pd.read_csv(OutputDir / "TrackSummary.csv"))
Path = OutputDir / "PreviewTracks.mp4"
if Path.exists():
    Mb = Path.stat().st_size / 2**20
    print(f"Preview: {Path} ({Mb:.0f} MB)")
    if Mb < 50:   # bigger files are too heavy to play inside the notebook; open them from Drive
        display(Video(str(Path), embed=True, width=960))

## About this step

**Model:** **SAM 2.1 Small in video mode** (Meta's Segment Anything 2.1, Ultralytics `SAM2VideoPredictor`, `sam2.1_s.pt`, ~46 M parameters). It is started from YOLO boxes (step 02) and follows the animal's outline using a memory of how it looked. Not trained on our cows.

**What we found:**
- First run (1080–1980 s, 5 fps, YOLO inside the same step; stopped after 45 min): **one track for all 900 s**, YOLO agreed on **97 %** of frames.
- Test 1080–1320 s at 2 fps (YOLO at 1024): **one track for all 240 s**; YOLO agrees 90 %, NoYoloBox 10 % (SAM covered frames YOLO missed), Disagrees 0 %. ~155 s per 2-minute chunk on a T4 (~1.5 frames/s).
- First full run: chunk 8 (14:00–16:00) was **skipped** and chunks 7 and 9 started late, because YOLO was never sure there and every chunk restarted from YOLO. Fixed: chunks now carry the cow over from the previous chunk. To test that stretch, set `TestStart = 720` in steps 02a/03a.

**Still true:** a TrackId is not a cow. A new track starts when a chunk cannot be joined to the previous one (she moved a lot at the border, or no confident YOLO box at a chunk start), and a calf can get its own track once YOLO boxes it.

Full write-up, glossary and all models: `CalvingPen/README.md` in the repo.